# Task 2 — кластеризация и поиск · 20 баллов

Статус: проектирование. Источник — PDF, страницы 3–5. Зависит от best custom checkpoint из Task 1.
Требование запрещает использовать labels для извлечения признаков и clustering, одновременно требуя supervised custom CNN из Task 1. Рабочая трактовка: заморозить ранее обученную CNN, в Task 2 не передавать labels в извлечение, preprocessing или выбор кластеров. Явно раскрыть supervised происхождение encoder.

# 1. Data Reading, Exploration and Preprocessing · 2 балла

Предлагаемый протокол: trainval — набор для clustering и gallery retrieval, test — только query retrieval. PDF не фиксирует gallery, поэтому этот выбор нужно явно описать.
Отдельные preprocessing pipelines для каждого extractor. Сохранить image IDs и порядок строк embeddings. Все обучаемые преобразования fit только на trainval.

# 2. Feature Extraction · 4 балла

Три замороженных extractor в eval/no_grad:
1. Best custom CNN, скрытое представление перед финальным classifier, предполагаемый размер 512.
2. ImageNet-supervised ResNet-50 без classification head, global pooled embedding.
3. Self-supervised DINOv2 ViT-S/14, например через torch.hub, embedding изображения.
Сохранить источник и версию весов, preprocessing и размеры features. Для второго extractor использовать исходные ImageNet weights, чтобы отдельно сравнить представления.
Cache embeddings вместе с IDs и конфигурацией. Для cosine retrieval выполнить L2 normalization. Если используется PCA для clustering, обучить на trainval и одинаково применять к сравниваемым алгоритмам.

# 2.1. Clustering · 5 баллов

Для каждого extractor выполнить K-means и предлагаемый GaussianMixture с diagonal covariance. Итого ≥6 комбинаций.
Предлагаемая сетка K включает 2,4,8,12,20,30,37,45,60. Построить elbow по inertia K-means, silhouette по каждому кандидату; для GMM отдельно выбирать K по silhouette. Описать правило разрешения расхождений между elbow и silhouette.
Сетка является стартовым планом, расширять при необходимости по внутренним метрикам. Labels не участвуют в выборе K. Обсудить совпадение с 37 породами и 2 видами.
При дорогом silhouette использовать фиксированную подвыборку и указать размер. Проверять, что в ней представлены ≥2 кластера.

# 3. Cluster Evaluation and Visualization · 6 баллов

Для всех комбинаций: silhouette, Davies–Bouldin, ARI, NMI, purity в общей таблице. Внешние метрики вычислять только после фиксации clustering. Purity = сумма максимального числа объектов одной породы в каждом кластере / N; обсудить зависимость от числа кластеров.
Предлагаемый выбор лучшего extractor для визуализации — по silhouette, раскрыть выбранный критерий.
PCA 2D и t-SNE либо UMAP, цвет соответствует cluster ID. Проекции служат для отображения. Показать сетки изображений минимум из 4 кластеров и интерпретации. Если оптимальное K=2, отдельно показать вариант K≥4, явно обозначив его дополнительным анализом.

# 3.1. Image Retrieval · 3 балла

retrieve(query_embedding, gallery_embeddings, k) возвращает IDs, similarity и изображения top-k по cosine similarity.
Использовать test queries и trainval gallery для всех трёх extractor. Релевантность определяется одинаковой породой. Само изображение исключать из выдачи, если протокол будет изменён на общую gallery.
Precision@k = число релевантных в top-k / k, итог — среднее по test queries. Отчёт для k∈{1,5,10} по каждому extractor. Показать 3 одинаковых query для сравнения экстракторов.
Считать similarity батчами при необходимости, проверять норму embeddings и стабильность обработки равных score.

# 4. Conclusion and Possible Improvements

Сопоставить clustering и retrieval, supervised и self-supervised представления. Указать ограничения внешних метрик и supervised custom encoder. Все таблицы и изображения должны быть сохранены в выполненном ноутбуке.